<a href="https://colab.research.google.com/github/greek-nlp/benchmark/blob/main/suite_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Greek NLP Benchmark Suite in Colab

This notebook is the Colab entrypoint for the benchmark tasks in this repository.

Current state:
- The notebook sets up Ollama inside Colab.
- It organizes the full task suite in one place.
- The repository now includes a shared runner in `suite_benchmark.py`.
- Direct shared-runner tasks currently include GEC, toxicity, machine translation, intent classification, and summarization.
- The remaining tasks still point to the task-specific notebooks already present in the repo.


## Quick Start

Run the notebook top to bottom:
1. Setup cells
2. Suite configuration
3. Model pull cell
4. Task overview
5. GEC benchmark cells

Important: in Ollama, use `llama3.1:8b`, not `llama3.1:8b-instruct`.


## Repo Setup

Open this notebook from the repository in Colab, or clone/upload the repository files into the runtime before continuing.


## 1. Environment Setup

In [1]:
!apt-get -qq update
!apt-get -qq install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!pip -q install pandas pywer openpyxl datasets scikit-learn

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print("Started Ollama server with PID", ollama_process.pid)

Started Ollama server with PID 19061


## 2. Suite Configuration

In [5]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
print("Working directory:", project_root)
print("Python executable:", sys.executable)

MODEL_PRESETS = {
    "t4": [
        "qwen2.5:3b",
        "gemma2:2b",
        "mistral:7b",
        "qwen2.5:7b-instruct",
    ],
    "high_memory": [
        "llama3.1:8b",
        "aya-expanse:8b",
        "gemma2:9b",
        "qwen2.5:14b",
    ],
}

TASKS = {
    "toxicity": {
        "datasets": ["OGTD / OffenseEval Greek"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "gec": {
        "datasets": ["GNC / Korre"],
        "entrypoints": ["suite_benchmark_colab.ipynb", "gec_benchmark_notebook.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "machine_translation": {
        "datasets": ["PGV"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "summarization": {
        "datasets": ["GreekLegalSum"],
        "entrypoints": ["nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "intent_classification": {
        "datasets": ["UniWay GR"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "shared_runner",
    },
    "ner": {
        "datasets": ["elNER", "ner_nel_greek_dataset", "UniWay GR"],
        "entrypoints": ["air_gr_nlp_benchmark.ipynb", "nlp_gr_access_data.ipynb"],
        "runner": "notebook",
    },
    "pos_tagging": {
        "datasets": ["UD Greek GDT"],
        "entrypoints": ["nlp_gr_access_data.ipynb"],
        "runner": "notebook",
    },
    "clustering": {
        "datasets": ["Greek Legal Code"],
        "entrypoints": ["nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
    "authorship_attribution": {
        "datasets": ["author datasets in repo notebooks"],
        "entrypoints": ["gr_attribution_zeroshot.ipynb", "nlp_gr_experiments.ipynb"],
        "runner": "notebook",
    },
}

MODEL_PRESET = "t4"
MODELS = MODEL_PRESETS[MODEL_PRESET]
ACTIVE_TASKS = list(TASKS)

SAMPLE_SIZE = 10
RANDOM_STATE = 42
OUTPUT_ROOT = Path("results/colab_suite")

print("Model preset:", MODEL_PRESET)
print("Models:", MODELS)
print("Tasks:", ACTIVE_TASKS)

Working directory: /content
Python executable: /usr/bin/python3
Model preset: t4
Models: ['qwen2.5:3b', 'gemma2:2b', 'mistral:7b', 'qwen2.5:7b-instruct']
Tasks: ['toxicity', 'gec', 'machine_translation', 'summarization', 'intent_classification', 'ner', 'pos_tagging', 'clustering', 'authorship_attribution']


In [6]:
for model in MODELS:
    print(f"Pulling {model}...")
    subprocess.run(["ollama", "pull", model], check=True)

Pulling qwen2.5:3b...
Pulling gemma2:2b...
Pulling mistral:7b...
Pulling qwen2.5:7b-instruct...


In [7]:
!ollama list

NAME                   ID              SIZE      MODIFIED               
qwen2.5:7b-instruct    845dbda0ea48    4.7 GB    Less than a second ago    
mistral:7b             6577803aa9a0    4.4 GB    35 seconds ago            
gemma2:2b              8ccf136fdd52    1.6 GB    About a minute ago        
qwen2.5:3b             357c53fb659c    1.9 GB    About a minute ago        


## 3. Task Overview

In [8]:
task_rows = []
for task_name, info in TASKS.items():
    task_rows.append(
        {
            "task": task_name,
            "runner": info["runner"],
            "datasets": ", ".join(info["datasets"]),
            "entrypoints": ", ".join(info["entrypoints"]),
        }
    )

pd.DataFrame(task_rows)

,task,runner,datasets,entrypoints
0,toxicity,shared_runner,OGTD / OffenseEval Greek,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
1,gec,shared_runner,GNC / Korre,"suite_benchmark_colab.ipynb, gec_benchmark_not..."
2,machine_translation,shared_runner,PGV,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
3,summarization,shared_runner,GreekLegalSum,nlp_gr_experiments.ipynb
4,intent_classification,shared_runner,UniWay GR,"air_gr_nlp_benchmark.ipynb, nlp_gr_experiments..."
5,ner,notebook,"elNER, ner_nel_greek_dataset, UniWay GR","air_gr_nlp_benchmark.ipynb, nlp_gr_access_data..."
6,pos_tagging,notebook,UD Greek GDT,nlp_gr_access_data.ipynb
7,clustering,notebook,Greek Legal Code,nlp_gr_experiments.ipynb
8,authorship_attribution,notebook,author datasets in repo notebooks,"gr_attribution_zeroshot.ipynb, nlp_gr_experime..."


## 4. Notebook Routing for Non-Shared Tasks

The shared runner currently covers GEC, toxicity, machine translation, intent classification, and summarization.

For the remaining tasks, start from these notebooks:
- NER: `air_gr_nlp_benchmark.ipynb`, `nlp_gr_access_data.ipynb`
- POS tagging: `nlp_gr_access_data.ipynb`
- Clustering: `nlp_gr_experiments.ipynb`
- Authorship attribution: `gr_attribution_zeroshot.ipynb`, `nlp_gr_experiments.ipynb`


## 5. Runnable Benchmark Example: GEC

In [10]:
import os

repo_url = "https://github.com/greek-nlp/benchmark.git"
repo_dir = "benchmark"

# Clone the repository if it doesn't already exist
if not os.path.exists(repo_dir):
    print(f"Cloning {repo_url} into {repo_dir}...")
    !git clone {repo_url}
    # Change current working directory to the cloned repository
    %cd {repo_dir}
    print(f"Changed current working directory to {os.getcwd()}")
else:
    print(f"Repository {repo_dir} already exists. Pulling latest changes...")
    # Change into the directory and pull latest changes
    %cd {repo_dir}
    !git pull
    print(f"Current working directory is {os.getcwd()}")

Cloning https://github.com/greek-nlp/benchmark.git into benchmark...
Cloning into 'benchmark'...
remote: Enumerating objects: 284, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (130/130), done.
remote: Total 284 (delta 76), reused 22 (delta 8), pack-reused 146 (from 1)
Receiving objects: 100% (284/284), 20.39 MiB | 10.30 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/benchmark
Changed current working directory to /content/benchmark


In [11]:
import sys
# Assuming project_root is defined in the previous setup cell (`suite-config`).
# If not, it would need to be re-initialized, e.g., project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from benchmark_suite import GenerationConfig, run_task, save_run_outputs

TASK_NAME = "gec"
TASK_OUTPUT_DIR = OUTPUT_ROOT / TASK_NAME
CONFIG = GenerationConfig(
    temperature=0.0,
    num_predict=256,
    timeout_seconds=300,
)

print("Running task:", TASK_NAME)
print("Models:", MODELS)
print("Output directory:", TASK_OUTPUT_DIR)

Running task: gec
Models: ['qwen2.5:3b', 'gemma2:2b', 'mistral:7b', 'qwen2.5:7b-instruct']
Output directory: results/colab_suite/gec


In [12]:
summary, raw = run_task(
    task_name=TASK_NAME,
    models=MODELS,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    config=CONFIG,
)

summary

,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds,task
0,qwen2.5:3b,10,0.0,11.403509,5.858855,9.734513,5.157401,0.9,10.261500,gec
3,qwen2.5:7b-instruct,10,0.2,13.157895,6.258322,11.946903,5.626256,0.9,8.821959,gec
1,gemma2:2b,10,0.1,32.456140,30.426099,30.088496,29.738781,0.8,7.223657,gec
2,mistral:7b,10,0.0,53.947368,47.470040,53.097345,47.019424,1.0,9.749548,gec


In [13]:
save_run_outputs(summary, raw, TASK_OUTPUT_DIR, TASK_NAME)
print(f"Saved benchmark outputs to {TASK_OUTPUT_DIR}")
summary

Saved benchmark outputs to results/colab_suite/gec


,model,samples,exact_match,wer_vs_reference,cer_vs_reference,wer_vs_original,cer_vs_original,changed_input_rate,avg_latency_seconds,task
0,qwen2.5:3b,10,0.0,11.403509,5.858855,9.734513,5.157401,0.9,10.261500,gec
3,qwen2.5:7b-instruct,10,0.2,13.157895,6.258322,11.946903,5.626256,0.9,8.821959,gec
1,gemma2:2b,10,0.1,32.456140,30.426099,30.088496,29.738781,0.8,7.223657,gec
2,mistral:7b,10,0.0,53.947368,47.470040,53.097345,47.019424,1.0,9.749548,gec
